In [1]:
import os

REPO_URL = "https://github.com/sinh2206/O_D.git"
REPO_DIR = "O_D"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch --all --prune
    !git checkout -B $(git symbolic-ref --short refs/remotes/origin/HEAD | sed 's@^origin/@@')
    !git reset --hard origin/$(git symbolic-ref --short refs/remotes/origin/HEAD | sed 's@^origin/@@')
    %cd ..

%cd {REPO_DIR}
!git log -1 --oneline


Cloning into 'O_D'...
remote: Enumerating objects: 9478, done.
remote: Counting objects: 100% (447/447), done.
remote: Compressing objects: 100% (370/370), done.
remote: Total 9478 (delta 190), reused 325 (delta 75), pack-reused 9031 (from 2)
Receiving objects: 100% (9478/9478), 1009.95 MiB | 34.62 MiB/s, done.
Resolving deltas: 100% (191/191), done.
Updating files: 100% (9023/9023), done.
/kaggle/working/O_D
9cd14dd (HEAD -> main, origin/main, origin/HEAD) 1.9.8


In [2]:
!pwd
!ls


/kaggle/working/O_D
img_error.py  models	       predict.py  README.md	     results   utils
LICENSE       obj-detec.ipynb  public	   requirements.txt  train.py


In [3]:
%pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 103.8 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 req

In [4]:
!python train.py \
  --train_data ./public/annotations/train.json \
  --val_data ./public/annotations/val.json \
  --image_dir ./public/train/images \
  --val_image_dir ./public/val/images \
  --checkpoint_dir ./models/

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth
100%|███████████████████████████████████████| 83.3M/83.3M [00:00<00:00, 181MB/s]
/kaggle/working/O_D/train.py:533: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)
Device: cuda, AMP: True
Train samples: 7500, Val samples: 1500, Classes: ['person', 'car', 'dog', 'cat', 'chair']
Balanced sampling: True
Class weights enabled: True
Class weights: [1.25, 1.05, 1.0, 1.0, 1.3]
Epoch 001/020 | train_loss=1.5106 (cls=0.4450, reg=0.7248, ctr=0.6818) | val_loss=1.2932
Saved best checkpoint: models/best.pth (val_loss=1.2932)
Epoch 002/020 | train_loss=1.1414 (cls=0.3195, reg=0.4933, ctr=0.6572) | val_loss=1.1090
Saved best checkpoint: models/best.pth (val_loss=1.1090)
Epoch 003/020 | train_loss=0.9734 (cls=0.2733, reg=0.3777, ctr=0.6

In [5]:
!python predict.py \
  --image_dir ./public/val/images \
  --output val_predictions.json \
  --model_path ./models/best.pth

Device: cuda
Predicted images: 1500
Saved JSON: val_predictions.json
Hardcase items: 50
Hardcase summary: results/hardcase_summary.json


In [6]:
!python public/tools/evaluate_predictions.py \
  --ground_truth public/annotations/val.json \
  --predictions val_predictions.json \
  --output val_score.json

{
  "mAP@0.5": 0.643355,
  "performance_points": 20,
  "iou_threshold": 0.5,
  "num_ground_truth_boxes": 2021,
  "num_predictions": 2454,
  "micro_precision": 0.599022,
  "micro_recall": 0.727363,
  "per_class": {
    "person": {
      "ap": 0.729566,
      "num_ground_truth": 1074,
      "num_predictions": 1249,
      "true_positives": 833,
      "false_positives": 416,
      "recall": 0.775605,
      "precision": 0.666934
    },
    "car": {
      "ap": 0.56078,
      "num_ground_truth": 283,
      "num_predictions": 353,
      "true_positives": 179,
      "false_positives": 174,
      "recall": 0.632509,
      "precision": 0.507082
    },
    "dog": {
      "ap": 0.711756,
      "num_ground_truth": 206,
      "num_predictions": 258,
      "true_positives": 158,
      "false_positives": 100,
      "recall": 0.76699,
      "precision": 0.612403
    },
    "cat": {
      "ap": 0.803272,
      "num_ground_truth": 176,
      "num_predictions": 207,
      "true_positives": 146,
      "fal

In [7]:
!python img_error.py

Rendered hardcase images: 50
Source summary: results/hardcase_summary.json
Predictions: val_predictions.json
Output dir: results


In [8]:
# Zip project excluding public
import os
from pathlib import Path

src = Path('/kaggle/working/O_D')
out_zip = Path('/kaggle/working/train.zip')

if not src.exists():
    raise FileNotFoundError(f'Not found: {src}')

cmd = f"cd {src} && zip -r {out_zip} . -x 'public/*' 'public/**'"
print(cmd)
ret = os.system(cmd)
if ret != 0:
    raise RuntimeError(f'zip failed with code {ret}')
print(f'Created: {out_zip}')


cd /kaggle/working/O_D && zip -r /kaggle/working/train.zip . -x 'public/*' 'public/**'
  adding: __pycache__/ (stored 0%)
  adding: __pycache__/predict.cpython-312.pyc (deflated 53%)
  adding: predict.py (deflated 74%)
  adding: val_score.json (deflated 72%)
  adding: models/ (stored 0%)
  adding: models/README.md (stored 0%)
  adding: models/last.pth (deflated 8%)
  adding: models/best.pth (deflated 8%)
  adding: LICENSE (deflated 41%)
  adding: .gitignore (deflated 10%)
  adding: val_predictions.json (deflated 90%)
  adding: README.md (stored 0%)
  adding: obj-detec.ipynb (deflated 80%)
  adding: train.py (deflated 72%)
  adding: requirements.txt (deflated 13%)
  adding: results/ (stored 0%)
  adding: results/hardcase_031_img_91a924e89a9c.jpg (deflated 1%)
  adding: results/hardcase_019_img_26de71d51856.jpg (deflated 0%)
  adding: results/hardcase_027_img_4f0c109f0aed.jpg (deflated 1%)
  adding: results/hardcase_036_img_dcbc7d08a6a4.jpg (deflated 0%)
  adding: results/hardcase_023_im